In [ ]:
!mkdir -p /kaggle/working/mouse-dynamics
!cp -r /kaggle/input/mouse-dynamics-ic/* /kaggle/working/mouse-dynamics/

In [ ]:
%pip install optuna fastparquet --quiet

In [ ]:
import sys
sys.path.insert(0, '/kaggle/working/mouse-dynamics')

In [ ]:
import os

os.makedirs('/kaggle/working/datasets/raw', exist_ok=True)

for dataset in ['balabit', 'bogazici', 'minecraft']:
    src = f'/kaggle/input/mouse-dynamics-{dataset}/{dataset}'
    dst = f'/kaggle/working/datasets/raw/{dataset}'
    if not os.path.exists(dst):
        os.symlink(src, dst)

In [ ]:
import os
os.makedirs('/kaggle/working/mouse-dynamics/optuna', exist_ok=True)

In [ ]:
from src.classifiers import EnumClassifiers
from src.dataset_loaders import EnumDatasets
from src.orchestrator import Orchestrator
from src.preprocessors import EnumPreprocessors
from src.splitters import EnumSplitters

ALL_CLASSIFIERS = [EnumClassifiers.RANDOM_FOREST, EnumClassifiers.MLP, EnumClassifiers.KNN]
ALL_WINDOW_SIZES = [10, 50, 100, 150]

ALL_COMBINATIONS = [
    (classifier, window_size)
    for classifier in ALL_CLASSIFIERS
    for window_size in ALL_WINDOW_SIZES
]

for dataset, classifier, window_size in ALL_COMBINATIONS:
    print("=" * 80)
    print(f"Starting analysis for {classifier.value}: [window_size={window_size}]")
    print("=" * 80)
        
    orchestrator = Orchestrator(
        dataset=EnumDatasets.BALABIT,
        splitter=EnumSplitters.HALF,
        classifier=classifier,
        preprocessor_window_size=window_size,
        preprocessor=EnumPreprocessors.KHAN,
        is_debug=False
    )

    orchestrator.run()
    
    print("=" * 80)
    print(f"Ending analysis for {classifier.value}: [window_size={window_size}]")
    print("=" * 80)